In [103]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

mobile_phone_use_backup = pd.read_csv("training_table.csv")

In [104]:
print(mobile_phone_use_backup['screen_state'].value_counts())
print("*"*50)
print(mobile_phone_use_backup['location_type'].value_counts())
print("*"*50)
print(mobile_phone_use_backup['activity_type'].value_counts())
print("*"*50)
print(mobile_phone_use_backup['last_app'].value_counts())
print("*"*50)
print("*"*50)

screen_state
off         71510
unknown     22683
unlocked    15658
on           3152
Name: count, dtype: int64
**************************************************
location_type
home        58372
noloc       33809
work        10144
repeated     7216
unknown      1301
single       1090
passing      1071
Name: count, dtype: int64
**************************************************
activity_type
still         58139
unknown       41921
tilting        4923
in_vehicle     4773
on_foot        3073
on_bicycle      174
Name: count, dtype: int64
**************************************************
last_app
unknown                         34336
com.sec.android.app.launcher    15549
com.whatsapp                     7322
com.android.launcher3            6644
com.android.keyguard             6300
                                ...  
net.languagecourse.vt.en            1
com.vacationrentals.homeaway        1
com.funeasylearn.english            1
com.cyngn.browser                   1
com.aftership.aftersh

In [154]:
mobile_phone_use = mobile_phone_use_backup.copy()
print("*"*50)
mobile_phone_use['location_type'] = mobile_phone_use['location_type'].str.replace("noloc", "unknown")
print(mobile_phone_use['location_type'].value_counts())

mobile_phone_use = mobile_phone_use.drop(columns=['hour', 'notif_time'])

**************************************************
location_type
home        58372
unknown     35110
work        10144
repeated     7216
single       1090
passing      1071
Name: count, dtype: int64


In [155]:
# Time slot encoding
time_slots = ['early morning', 'mid morning', 'late morning', 'afternoon', 'evening', 'night']
mobile_phone_use['time_slot'] = pd.Categorical(mobile_phone_use['time_slot'], categories = time_slots)
mobile_phone_use = pd.get_dummies(mobile_phone_use, columns=['time_slot'], drop_first=True)

In [156]:
# Screen state encoding
custom_order = ['unknown', 'off', 'unlocked', 'on']
mobile_phone_use['screen_state'] = pd.Categorical(mobile_phone_use['screen_state'], categories = custom_order)
mobile_phone_use = pd.get_dummies(mobile_phone_use, columns=['screen_state'], drop_first=True)

In [157]:
# Location type encoding
mobile_phone_use['location_type'] = mobile_phone_use['location_type'].str.replace(['repeated', 'single', 'passing'], 'other')
custom_order = ['unknown', 'home', 'work', 'other']
mobile_phone_use['location_type'] = pd.Categorical(mobile_phone_use['location_type'], categories = custom_order)
mobile_phone_use = pd.get_dummies(mobile_phone_use, columns=['location_type'], drop_first=True)

In [158]:
# Activity type encoding
custom_order = ['unknown', 'still', 'tilting', 'in_vehicle', 'on_foot', 'on_bicycle']
mobile_phone_use['activity_type'] = pd.Categorical(mobile_phone_use['activity_type'], categories = custom_order)
mobile_phone_use = pd.get_dummies(mobile_phone_use, columns=['activity_type'], drop_first=True)

In [159]:
# Last application encoding
app_counts = mobile_phone_use_backup['last_app'].value_counts()
frequent_apps = app_counts[app_counts >= 100].index
mobile_phone_use['last_app'] = mobile_phone_use['last_app'].where(mobile_phone_use['last_app'].isin(frequent_apps), 'other')

custom_order = mobile_phone_use['last_app'].value_counts().index.tolist()
mobile_phone_use['last_app'] = pd.Categorical(mobile_phone_use['last_app'], categories = custom_order)
mobile_phone_use = pd.get_dummies(mobile_phone_use, columns=['last_app'], drop_first=True)

In [160]:
bool_cols = mobile_phone_use.select_dtypes(include='bool').columns
mobile_phone_use[bool_cols] = mobile_phone_use[bool_cols].astype(int)

In [161]:
scale_cols = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'prev_notifications',
    'prev_response_rate',
    'recent_response_rate',
    'slot_prev_response_rate',
    'mins_since_prev_notification',
    'age'
]

scaler = StandardScaler()

mobile_phone_use[scale_cols] = scaler.fit_transform(mobile_phone_use[scale_cols])

In [162]:
print("Final shape:", mobile_phone_use.shape)
print("Users:", mobile_phone_use["uuid"].nunique())
print(mobile_phone_use["responded"].value_counts())
mobile_phone_use.to_csv("encoded_training_table.csv", index=False)

Final shape: (113003, 103)
Users: 342
responded
0    81265
1    31738
Name: count, dtype: int64
